# Artificial Intelligence Technology and Application

## Machine Learning Lab Guide - Student Version

Independent implementation prepared by **Sundetkhan Bekzat**.


# 1 Feature Engineering on Credit Data

This notebook keeps the lab objective but uses compact local examples so it can run without external datasets.


## 1.1 Credit Data Frame
The table intentionally contains missing values to exercise preprocessing.


In [1]:
import numpy as np
import pandas as pd

credit = pd.DataFrame({
    "age": [24, 31, 46, 52, np.nan, 38, 29, 43],
    "income": [180, 240, 510, 620, 390, np.nan, 210, 470],
    "married": [0, 1, 1, 1, 0, 1, np.nan, 1],
    "city": ["Astana", "Almaty", "Astana", "Shymkent", "Astana", "Almaty", "Astana", "Shymkent"],
    "defaulted": [0, 0, 1, 1, 0, 1, 0, 1],
})
print(credit)
print("missing ratios:")
print(credit.isna().mean())


    age  income  married      city  defaulted
0  24.0   180.0      0.0    Astana          0
1  31.0   240.0      1.0    Almaty          0
2  46.0   510.0      1.0    Astana          1
3  52.0   620.0      1.0  Shymkent          1
4   NaN   390.0      0.0    Astana          0
5  38.0     NaN      1.0    Almaty          1
6  29.0   210.0      NaN    Astana          0
7  43.0   470.0      1.0  Shymkent          1
missing ratios:
age          0.125
income       0.125
married      0.125
city         0.000
defaulted    0.000
dtype: float64


## 1.2 Missing Value Repair
Numeric fields are imputed with medians before statistical feature selection.


In [2]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

numeric = credit[["age", "income", "married"]]
imputed = SimpleImputer(strategy="median").fit_transform(numeric)
scaled = MinMaxScaler().fit_transform(imputed)
selector = SelectKBest(score_func=chi2, k=2).fit(scaled, credit["defaulted"])
print(dict(zip(numeric.columns, selector.scores_.round(3))))
print("selected:", list(numeric.columns[selector.get_support()]))


{'age': np.float64(1.065), 'income': np.float64(1.362), 'married': np.float64(0.667)}
selected: ['age', 'income']


## 1.3 Encoding and Construction
Categorical city values are one-hot encoded and income is expanded polynomially.


In [3]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures

encoded_city = pd.get_dummies(credit["city"], prefix="city", dtype=int)
income_ready = SimpleImputer(strategy="median").fit_transform(credit[["income"]])
income_poly = PolynomialFeatures(degree=2, include_bias=False).fit_transform(income_ready)
engineered = pd.concat([
    pd.DataFrame(imputed, columns=["age", "income", "married"]),
    encoded_city.reset_index(drop=True),
    pd.DataFrame(income_poly[:, 1:], columns=["income_squared"]),
], axis=1)
print(engineered.head())


    age  income  married  city_Almaty  city_Astana  city_Shymkent  \
0  24.0   180.0      0.0            0            1              0   
1  31.0   240.0      1.0            1            0              0   
2  46.0   510.0      1.0            0            1              0   
3  52.0   620.0      1.0            0            0              1   
4  38.0   390.0      0.0            0            1              0   

   income_squared  
0         32400.0  
1         57600.0  
2        260100.0  
3        384400.0  
4        152100.0  


## 1.4 Wrapper Selection
Recursive feature elimination is demonstrated with a small logistic model.


In [4]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

wrapper = RFE(LogisticRegression(max_iter=500), n_features_to_select=3)
wrapper.fit(engineered, credit["defaulted"])
print("RFE selected:", list(engineered.columns[wrapper.support_]))


RFE selected: ['married', 'city_Almaty', 'city_Astana']
